In [ ]:
import math

ISA_SEA_LEVEL_TEMPERATURE_K = 288.15
ISA_SEA_LEVEL_PRESSURE_PA = 101_325.0
GRAVITY_M_S2 = 9.80665
ISA_TROPOSPHERE_LAPSE_RATE_K_M = 0.0065
FEET_TO_METERS = 0.3048
KNOTS_TO_METERS_PER_SECOND = 1852.0 / 3600.0
GAMMA = 1.4
GAS_CONSTANT_J_KG_K = 287.05287


# ISA pressure at altitude; the DISA offset changes local temperature and density.
def air_density(altitude_ft, temperature_k):
    altitude_m = altitude_ft * FEET_TO_METERS
    isa_temperature_k = ISA_SEA_LEVEL_TEMPERATURE_K - ISA_TROPOSPHERE_LAPSE_RATE_K_M * altitude_m
    pressure_pa = ISA_SEA_LEVEL_PRESSURE_PA * (
        isa_temperature_k / ISA_SEA_LEVEL_TEMPERATURE_K
    ) ** (GRAVITY_M_S2 / (GAS_CONSTANT_J_KG_K * ISA_TROPOSPHERE_LAPSE_RATE_K_M))
    return pressure_pa / (GAS_CONSTANT_J_KG_K * temperature_k)


def calculate_tip_mach(name, diameter_m, rpm, altitude_ft, delta_isa_c,
                       true_airspeed_knots, angle_of_attack_deg):
    altitude_m = altitude_ft * FEET_TO_METERS
    isa_temperature_k = ISA_SEA_LEVEL_TEMPERATURE_K - ISA_TROPOSPHERE_LAPSE_RATE_K_M * altitude_m
    temperature_k = isa_temperature_k + delta_isa_c
    speed_of_sound_m_s = math.sqrt(GAMMA * GAS_CONSTANT_J_KG_K * temperature_k)
    density_kg_m3 = air_density(altitude_ft, temperature_k)
    revolutions_per_second = rpm / 60.0
    true_airspeed_m_s = true_airspeed_knots * KNOTS_TO_METERS_PER_SECOND
    axial_airspeed_m_s = true_airspeed_m_s * math.cos(math.radians(angle_of_attack_deg))
    rotational_tip_speed_m_s = math.pi * diameter_m * revolutions_per_second
    helical_tip_speed_m_s = math.hypot(rotational_tip_speed_m_s, axial_airspeed_m_s)
    tip_dynamic_pressure_pa = 0.5 * density_kg_m3 * helical_tip_speed_m_s**2
    return {
        'name': name,
        'temperature_k': temperature_k,
        'density_kg_m3': density_kg_m3,
        'speed_of_sound_m_s': speed_of_sound_m_s,
        'rotational_tip_speed_m_s': rotational_tip_speed_m_s,
        'axial_airspeed_m_s': axial_airspeed_m_s,
        'helical_tip_speed_m_s': helical_tip_speed_m_s,
        'tip_dynamic_pressure_pa': tip_dynamic_pressure_pa,
        'rotational_tip_mach': rotational_tip_speed_m_s / speed_of_sound_m_s,
        'helical_tip_mach': helical_tip_speed_m_s / speed_of_sound_m_s,
        'advance_ratio': true_airspeed_m_s / (revolutions_per_second * diameter_m),
    }


propellers = [
    {
        'name': 'POC2',
        'diameter_m': 2.8,
        'tip_chord_m': 0.103,
        'target_y+': 18,
        'rpm': 1000.0,
        'altitude_ft': 2460.0,
        'delta_isa_c': 20.0,
        'true_airspeed_knots': 0.0,
        'angle_of_attack_deg': 0.0,
        'thrust_n': 4195.0,
    },
    {
        'name': 'Pusher',
        'diameter_m': 2.120,
        'tip_chord_m': 0.056,
        'target_y+': 18,
        'rpm': 2186.0,
        'altitude_ft': 0.0,
        'delta_isa_c': 10.0,
        'true_airspeed_knots': 91.6,
        'angle_of_attack_deg': 3.5,
        'thrust_n': 4914.0,
    },
]

results = {
    case['name']: calculate_tip_mach(
        **{key: value for key, value in case.items() if key not in ('tip_chord_m', 'target_y+', 'thrust_n')}
    )
    for case in propellers
}
for name, result in results.items():
    print(name)
    print(f"  Air temperature: {result['temperature_k']:.2f} K; speed of sound: {result['speed_of_sound_m_s']:.2f} m/s")
    print(f"  Air density: {result['density_kg_m3']:.4f} kg/m³")
    print(f"  Rotational tip speed: {result['rotational_tip_speed_m_s']:.2f} m/s")
    print(f"  Axial airspeed: {result['axial_airspeed_m_s']:.2f} m/s")
    print(f"  Helical tip speed: {result['helical_tip_speed_m_s']:.2f} m/s")
    print(f"  Helical-tip dynamic pressure: {result['tip_dynamic_pressure_pa']:.2f} Pa "
          f"({result['tip_dynamic_pressure_pa'] / 1_000:.3f} kPa)")
    print(f"  Helical tip Mach: {result['helical_tip_mach']:.3f}")
    print(f"  Rotation-only tip Mach: {result['rotational_tip_mach']:.3f}")
    print(f"  Advance ratio: {result['advance_ratio']:.3f}")
    print()
print('Pusher supplied advance ratio (0.61) agrees with the calculated value.')

In [ ]:
# 10 kHz design frequency from the acoustic-mesh-resolution note.
frequency_hz = 10_000.0
samples_per_wavelength = (8, 16)
millimeters_per_meter = 1_000.0

# The note gives separate acoustic (c/f) and convective (U_c/f) criteria.
# As assumed here, U_c is the rotational tip speed. A tip cell should meet
# both criteria, so use the smaller maximum spacing at a given PPW.
mesh_results = {}
for name, result in results.items():
    acoustic_wavelength_m = result['speed_of_sound_m_s'] / frequency_hz
    convective_wavelength_m = result['rotational_tip_speed_m_s'] / frequency_hz
    spacings_m = {}
    for ppw in samples_per_wavelength:
        acoustic_spacing_m = acoustic_wavelength_m / ppw
        convective_spacing_m = convective_wavelength_m / ppw
        spacings_m[ppw] = {
            'acoustic': acoustic_spacing_m,
            'convective': convective_spacing_m,
            'tip': min(acoustic_spacing_m, convective_spacing_m),
        }
    mesh_results[name] = spacings_m

    print(f'{name} at {frequency_hz:,.0f} Hz')
    print(f'  Acoustic wavelength (c/f): {acoustic_wavelength_m * millimeters_per_meter:.2f} mm')
    print(f'  Convective wavelength (U_tip/f): {convective_wavelength_m * millimeters_per_meter:.2f} mm')
    for ppw in (16, 8):
        spacing = spacings_m[ppw]
        print(
            f"  {ppw} PPW: acoustic limit {spacing['acoustic'] * millimeters_per_meter:.3f} mm; "
            f"tip-convective limit {spacing['convective'] * millimeters_per_meter:.3f} mm; "
            f"tip mesh spacing <= {spacing['tip'] * millimeters_per_meter:.3f} mm"
        )
    print(
        f"  Recommended tip spacing (16 PPW): <= "
        f"{spacings_m[16]['tip'] * millimeters_per_meter:.3f} mm"
    )
    print()

In [ ]:
from flow360_utils.flow360_tools import calculate_flat_plate_turbulent_y1

SUTHERLAND_REFERENCE_TEMPERATURE_K = 273.15
SUTHERLAND_REFERENCE_VISCOSITY_PA_S = 1.716e-5
SUTHERLAND_CONSTANT_K = 110.4
MICROMETERS_PER_METER = 1_000_000.0

def air_properties(altitude_ft, temperature_k):
    density_kg_m3 = air_density(altitude_ft, temperature_k)
    dynamic_viscosity_pa_s = SUTHERLAND_REFERENCE_VISCOSITY_PA_S * (
        temperature_k / SUTHERLAND_REFERENCE_TEMPERATURE_K
    ) ** 1.5 * (
        (SUTHERLAND_REFERENCE_TEMPERATURE_K + SUTHERLAND_CONSTANT_K)
        / (temperature_k + SUTHERLAND_CONSTANT_K)
    )
    return density_kg_m3, dynamic_viscosity_pa_s


y1_results = {}
for case in propellers:
    name = case['name']
    result = results[name]
    density_kg_m3, dynamic_viscosity_pa_s = air_properties(
        case['altitude_ft'], result['temperature_k']
    )
    # Helical tip speed approximates local flow relative to the blade tip.
    reynolds_number, y1_m = calculate_flat_plate_turbulent_y1(
        freestream_velocity=result['helical_tip_speed_m_s'],
        density=density_kg_m3,
        dynamic_viscosity=dynamic_viscosity_pa_s,
        boundary_layer_length=case['tip_chord_m'],
        desired_yplus=case['target_y+'],
    )
    y1_results[name] = {
        'reynolds_number': reynolds_number,
        'y1_m': y1_m,
        'density_kg_m3': density_kg_m3,
        'dynamic_viscosity_pa_s': dynamic_viscosity_pa_s,
    }
    print(f"{name}: tip chord {case['tip_chord_m']:.3f} m, target y+ {case['target_y+']:g}")
    print(f"  Tip-relative speed: {result['helical_tip_speed_m_s']:.2f} m/s")
    print(f"  Density: {density_kg_m3:.4f} kg/m³; dynamic viscosity: {dynamic_viscosity_pa_s:.3e} Pa·s")
    print(f"  Chord Reynolds number: {reynolds_number:,.0f}")
    print(f"  First-cell wall distance y1: {y1_m * MICROMETERS_PER_METER:.1f} µm ({y1_m * 1_000:.4f} mm)")
    print()

### Ideal propeller wake contraction (far wake)

For positive axial thrust, approximate the propeller as a one-dimensional, incompressible actuator disk with uniform loading. Let $R$ be blade radius, $T$ thrust, $\rho$ air density, $V_a$ incoming velocity along the propeller axis, $V_d$ velocity at the disk, and $V_w$ velocity in the fully developed far wake. For small incidence, use $V_a \approx V_\infty \cos\alpha$.

$$A=\pi R^2, \qquad \dot m=\rho A V_d$$
$$T=\dot m(V_w-V_a), \qquad V_d=\frac{V_a+V_w}{2}$$
$$V_w=\sqrt{V_a^2+\frac{2T}{\rho A}}, \qquad \frac{R_w}{R}=\sqrt{\frac{V_d}{V_w}}=\sqrt{\frac{V_a+V_w}{2V_w}}$$
Define the radial contraction as $C_r=1-R_w/R$.

Using the propeller diameters and atmospheric conditions already defined in this notebook, with the specified target thrusts:

| Case | $T$ (N) | $R$ (m) | $\rho$ (kg/m³) | $V_a$ (m/s) | $V_w$ (m/s) | $R_w/R$ | $R_w$ (m) | Radial contraction |
|:--|--:|--:|--:|--:|--:|--:|--:|--:|
| POC2 (Lifter; static) | 4,195 | 1.400 | 1.0641 | 0 | 35.784 | 0.7071 | 0.9899 | 29.29% |
| Pusher (91.6 kn, 3.5°) | 4,914 | 1.060 | 1.1839 | 47.035 | 67.558 | 0.9209 | 0.9762 | 7.91% |

The Pusher airspeed above is the configured flight condition. For a static condition ($V_a=0$), the ideal far-wake ratio is $R_w/R=1/\sqrt{2}=0.7071$. These equations give only the fully developed ideal wake; thrust and airspeed alone do not determine contraction at a particular downstream distance or the exact tip-vortex trajectory.

Source: [NASA Glenn Research Center, Propeller Thrust](https://www1.grc.nasa.gov/beginners-guide-to-aeronautics/propeller-thrust/).

In [ ]:
# Ideal, fully developed actuator-disk wake contraction for each propeller.
wake_contraction_results = {}
for case in propellers:
    name = case['name']
    thrust_n = case['thrust_n']
    radius_m = case['diameter_m'] / 2
    disk_area_m2 = math.pi * radius_m**2
    density_kg_m3, _ = air_properties(case['altitude_ft'], results[name]['temperature_k'])
    axial_airspeed_m_s = (
        case['true_airspeed_knots']
        * KNOTS_TO_METERS_PER_SECOND
        * math.cos(math.radians(case['angle_of_attack_deg']))
    )
    if thrust_n <= 0 or axial_airspeed_m_s < 0:
        raise ValueError(f"{name} requires positive thrust and nonnegative axial airspeed")

    far_wake_speed_m_s = math.sqrt(
        axial_airspeed_m_s**2 + 2 * thrust_n / (density_kg_m3 * disk_area_m2)
    )
    disk_speed_m_s = (axial_airspeed_m_s + far_wake_speed_m_s) / 2
    wake_radius_ratio = math.sqrt(disk_speed_m_s / far_wake_speed_m_s)
    wake_radius_m = radius_m * wake_radius_ratio
    radial_contraction = 1 - wake_radius_ratio
    wake_contraction_results[name] = {
        'thrust_n': thrust_n,
        'density_kg_m3': density_kg_m3,
        'axial_airspeed_m_s': axial_airspeed_m_s,
        'disk_speed_m_s': disk_speed_m_s,
        'far_wake_speed_m_s': far_wake_speed_m_s,
        'wake_radius_ratio': wake_radius_ratio,
        'wake_radius_m': wake_radius_m,
        'radial_contraction': radial_contraction,
    }
    print(
        f"{name}: T = {thrust_n:,.0f} N, rho = {density_kg_m3:.4f} kg/m³, "
        f"Va = {axial_airspeed_m_s:.3f} m/s, Vw = {far_wake_speed_m_s:.3f} m/s"
    )
    print(
        f"  Rw/R = {wake_radius_ratio:.4f}, Rw = {wake_radius_m:.4f} m, "
        f"radial contraction = {radial_contraction:.2%}"
    )

In [ ]:
# Temporal resolution depends on RPM and the target absolute frequency, not blade radius.
N_sample_temporal = 13.3333  # minimum time steps per period at frequency_hz
reference_steps_per_rev = 8_000  # previous POC2 setting for comparison

if frequency_hz <= 0 or N_sample_temporal <= 0:
    raise ValueError('frequency_hz and N_sample_temporal must be positive')

max_time_step_s = 1.0 / (N_sample_temporal * frequency_hz)
temporal_results = {}
for case in propellers:
    rpm = case['rpm']
    if rpm <= 0:
        raise ValueError(f"{case['name']} RPM must be positive")
    revolution_period_s = 60.0 / rpm
    exact_steps_per_rev = revolution_period_s / max_time_step_s
    min_steps_per_rev = math.ceil(exact_steps_per_rev)
    time_step_s = revolution_period_s / min_steps_per_rev
    degrees_per_step = 360.0 / min_steps_per_rev
    achieved_samples_per_period = 1.0 / (frequency_hz * time_step_s)
    temporal_results[case['name']] = {
        'min_steps_per_rev': min_steps_per_rev,
        'max_time_step_s': max_time_step_s,
        'actual_time_step_s': time_step_s,
        'degrees_per_step': degrees_per_step,
        'samples_per_period': achieved_samples_per_period,
    }
    print(f"{case['name']} at {rpm:,.0f} RPM and {frequency_hz:,.0f} Hz")
    print(f"  Revolution period: {revolution_period_s * 1_000:.6f} ms")
    print(f"  Maximum time step ({N_sample_temporal} samples/period): {max_time_step_s * 1_000_000:.3f} µs")
    print(f"  Minimum whole steps/rev: {min_steps_per_rev:,} (unrounded: {exact_steps_per_rev:,.3f})")
    print(f"  At {min_steps_per_rev:,} steps/rev: Δt = {time_step_s * 1_000_000:.6f} µs, "
          f"Δθ = {degrees_per_step:.4f}°/step, {achieved_samples_per_period:.3f} samples/period")
    print()

reference_case = next(case for case in propellers if case['name'] == 'POC2')
reference_period_s = 60.0 / reference_case['rpm']
reference_time_step_s = reference_period_s / reference_steps_per_rev
reference_samples_per_period = 1.0 / (frequency_hz * reference_time_step_s)
reference_bandwidth_hz = 1.0 / (N_sample_temporal * reference_time_step_s)
print(f"{reference_case['name']} at {reference_steps_per_rev:,} steps/rev:")
print(f"  Δt = {reference_time_step_s * 1_000_000:.3f} µs; "
      f"{reference_samples_per_period:.2f} samples at {frequency_hz:,.0f} Hz")
print(f"  Meets {N_sample_temporal} samples/period: "
      f"{reference_steps_per_rev >= temporal_results[reference_case['name']]['min_steps_per_rev']}")
print(f"  {N_sample_temporal}-sample bandwidth: {reference_bandwidth_hz / 1_000:.2f} kHz")